In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
REPO = "https://github.com/RAj5517/Molecular-Property-Predictor"
REPO_DIR = "/content/Molecular-Property-Predictor"
if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO}")
else:
    os.chdir(REPO_DIR)
    os.system("git pull origin main")
os.chdir(REPO_DIR)
print(f"Working in: {os.getcwd()}")

Working in: /content/Molecular-Property-Predictor


In [3]:
!pip install rdkit transformers torch scikit-learn huggingface_hub -q
print("Done ✓")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 37.9 MB/s eta 0:00:00
Done ✓


In [4]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


# ***Loading both datasets***

In [7]:
import pandas as pd
import numpy as np

# BBBP — load from URL
bbbp = pd.read_csv("https://raw.githubusercontent.com/GLambard/Molecules_Dataset_Collection/master/latest/BBBP.csv")
bbbp = bbbp.dropna(subset=['smiles'])
print(f"BBBP: {bbbp.shape}")
print(bbbp.columns.tolist())

# ESOL — already worked before
esol = pd.read_csv("https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv")
print(f"ESOL: {esol.shape}")
print(esol.columns.tolist())

BBBP: (2039, 5)
['Unnamed: 0', 'num', 'name', 'p_np', 'smiles']
ESOL: (1128, 10)
['Compound ID', 'ESOL predicted log solubility in mols per litre', 'Minimum Degree', 'Molecular Weight', 'Number of H-Bond Donors', 'Number of Rings', 'Number of Rotatable Bonds', 'Polar Surface Area', 'measured log solubility in mols per litre', 'smiles']


# ***Tokenizer + Dataset class for multi-task***

In [8]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")

class MultiTaskDataset(Dataset):
    def __init__(self, smiles_list, bbbp_label=None, esol_label=None, max_length=128):
        self.smiles = smiles_list
        self.bbbp   = bbbp_label    # 0 or 1 or None
        self.esol   = esol_label    # float or None
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.smiles[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        item = {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'bbbp_label':     torch.tensor(self.bbbp[idx], dtype=torch.float) if self.bbbp is not None else torch.tensor(-1.0),
            'esol_label':     torch.tensor(self.esol[idx], dtype=torch.float) if self.esol is not None else torch.tensor(float('nan')),
        }
        return item

# split both datasets
from sklearn.model_selection import train_test_split

# BBBP split
bbbp_train_s, bbbp_test_s, bbbp_train_l, bbbp_test_l = train_test_split(
    bbbp['smiles'].tolist(), bbbp['p_np'].tolist(),
    test_size=0.2, random_state=42, stratify=bbbp['p_np'].tolist()
)

# ESOL split
esol_train_s, esol_test_s, esol_train_l, esol_test_l = train_test_split(
    esol['smiles'].tolist(),
    esol['measured log solubility in mols per litre'].tolist(),
    test_size=0.2, random_state=42
)

# BBBP dataset — esol label is None
bbbp_train_ds = MultiTaskDataset(bbbp_train_s, bbbp_label=bbbp_train_l, esol_label=None)
bbbp_test_ds  = MultiTaskDataset(bbbp_test_s,  bbbp_label=bbbp_test_l,  esol_label=None)

# ESOL dataset — bbbp label is None
esol_train_ds = MultiTaskDataset(esol_train_s, bbbp_label=None, esol_label=esol_train_l)
esol_test_ds  = MultiTaskDataset(esol_test_s,  bbbp_label=None, esol_label=esol_test_l)

bbbp_train_loader = DataLoader(bbbp_train_ds, batch_size=32, shuffle=True)
bbbp_test_loader  = DataLoader(bbbp_test_ds,  batch_size=32, shuffle=False)
esol_train_loader = DataLoader(esol_train_ds, batch_size=32, shuffle=True)
esol_test_loader  = DataLoader(esol_test_ds,  batch_size=32, shuffle=False)

print(f"BBBP train: {len(bbbp_train_ds)} · test: {len(bbbp_test_ds)}")
print(f"ESOL train: {len(esol_train_ds)} · test: {len(esol_test_ds)}")
print("Datasets ready ✓")

tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

BBBP train: 1631 · test: 408
ESOL train: 902 · test: 226
Datasets ready ✓


# ***Multi-task model architecture***

In [6]:
import torch.nn as nn
from transformers import AutoModel

class MultiTaskChemBERTa(nn.Module):
    def __init__(self):
        super().__init__()
        # shared backbone — learns general molecular features
        self.backbone = AutoModel.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")

        hidden = self.backbone.config.hidden_size   # 768

        # task-specific heads
        self.bbbp_head = nn.Sequential(
            nn.Linear(hidden, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1)     # binary classification
        )
        self.esol_head = nn.Sequential(
            nn.Linear(hidden, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1)     # regression
        )

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]   # [CLS] token = molecule representation
        return self.bbbp_head(cls), self.esol_head(cls)

model = MultiTaskChemBERTa().to('cuda')
total = sum(p.numel() for p in model.parameters())
print(f"Multi-task model ready ✓")
print(f"Parameters: {total:,}")
print(f"Backbone: ChemBERTa (shared)")
print(f"Heads: BBBP classifier + ESOL regressor")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/501 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/179M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: seyonec/ChemBERTa-zinc-base-v1
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Multi-task model ready ✓
Parameters: 44,301,058
Backbone: ChemBERTa (shared)
Heads: BBBP classifier + ESOL regressor


# ***Train multi-task model***

In [9]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import roc_auc_score
from torch.nn import BCEWithLogitsLoss, MSELoss
import os

EPOCHS   = 6
LR       = 2e-5
SAVE_DIR = '/content/drive/MyDrive/mol_predictor/checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = (len(bbbp_train_loader) + len(esol_train_loader)) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

bce_loss = BCEWithLogitsLoss()
mse_loss = MSELoss()

best_bbbp_auc = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    steps = 0

    # alternate between BBBP and ESOL batches
    for bbbp_batch, esol_batch in zip(bbbp_train_loader, esol_train_loader):

        # ── BBBP step ──
        input_ids = bbbp_batch['input_ids'].to('cuda')
        attn_mask = bbbp_batch['attention_mask'].to('cuda')
        bbbp_labels = bbbp_batch['bbbp_label'].to('cuda')

        optimizer.zero_grad()
        bbbp_logits, _ = model(input_ids, attn_mask)
        loss_bbbp = bce_loss(bbbp_logits.squeeze(), bbbp_labels)
        loss_bbbp.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss_bbbp.item()

        # ── ESOL step ──
        input_ids = esol_batch['input_ids'].to('cuda')
        attn_mask = esol_batch['attention_mask'].to('cuda')
        esol_labels = esol_batch['esol_label'].to('cuda')

        optimizer.zero_grad()
        _, esol_preds = model(input_ids, attn_mask)
        loss_esol = mse_loss(esol_preds.squeeze(), esol_labels)
        loss_esol.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss_esol.item()
        steps += 2

    avg_loss = total_loss / steps

    # ── Evaluate BBBP ──
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in bbbp_test_loader:
            input_ids = batch['input_ids'].to('cuda')
            attn_mask = batch['attention_mask'].to('cuda')
            bbbp_logits, _ = model(input_ids, attn_mask)
            probs = torch.sigmoid(bbbp_logits.squeeze())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(batch['bbbp_label'].numpy())

    bbbp_auc = roc_auc_score(all_labels, all_probs)

    # ── Evaluate ESOL ──
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in esol_test_loader:
            input_ids = batch['input_ids'].to('cuda')
            attn_mask = batch['attention_mask'].to('cuda')
            _, esol_preds = model(input_ids, attn_mask)
            all_preds.extend(esol_preds.squeeze().cpu().numpy())
            all_targets.extend(batch['esol_label'].numpy())

    esol_rmse = np.sqrt(np.mean((np.array(all_preds) - np.array(all_targets))**2))

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | BBBP AUC: {bbbp_auc:.4f} | ESOL RMSE: {esol_rmse:.4f}")

    if bbbp_auc > best_bbbp_auc:
        best_bbbp_auc = bbbp_auc
        torch.save(model.state_dict(), f'{SAVE_DIR}/multitask_best.pt')
        print(f"  → Best saved (AUC: {best_bbbp_auc:.4f}) ✓")

print(f"\nBest BBBP AUC:  {best_bbbp_auc:.4f}")
print(f"ChemBERTa AUC:  0.9339")
print(f"RF Baseline:    0.9330")

Epoch 1/6 | Loss: 4.3909 | BBBP AUC: 0.8091 | ESOL RMSE: 1.9792
  → Best saved (AUC: 0.8091) ✓
Epoch 2/6 | Loss: 1.4687 | BBBP AUC: 0.8869 | ESOL RMSE: 1.3428
  → Best saved (AUC: 0.8869) ✓
Epoch 3/6 | Loss: 0.9584 | BBBP AUC: 0.9089 | ESOL RMSE: 1.1742
  → Best saved (AUC: 0.9089) ✓
Epoch 4/6 | Loss: 0.7736 | BBBP AUC: 0.9059 | ESOL RMSE: 1.0853
Epoch 5/6 | Loss: 0.6390 | BBBP AUC: 0.9164 | ESOL RMSE: 1.0351
  → Best saved (AUC: 0.9164) ✓
Epoch 6/6 | Loss: 0.5717 | BBBP AUC: 0.9210 | ESOL RMSE: 0.9791
  → Best saved (AUC: 0.9210) ✓

Best BBBP AUC:  0.9210
ChemBERTa AUC:  0.9339
RF Baseline:    0.9330


In [10]:
# continue training — 6 more epochs
EXTRA_EPOCHS = 6
best_bbbp_auc = 0.9210  # continue from where we left off

for epoch in range(EXTRA_EPOCHS):
    model.train()
    total_loss = 0
    steps = 0

    for bbbp_batch, esol_batch in zip(bbbp_train_loader, esol_train_loader):

        # BBBP step
        input_ids = bbbp_batch['input_ids'].to('cuda')
        attn_mask = bbbp_batch['attention_mask'].to('cuda')
        bbbp_labels = bbbp_batch['bbbp_label'].to('cuda')

        optimizer.zero_grad()
        bbbp_logits, _ = model(input_ids, attn_mask)
        loss_bbbp = bce_loss(bbbp_logits.squeeze(), bbbp_labels)
        loss_bbbp.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss_bbbp.item()

        # ESOL step
        input_ids = esol_batch['input_ids'].to('cuda')
        attn_mask = esol_batch['attention_mask'].to('cuda')
        esol_labels = esol_batch['esol_label'].to('cuda')

        optimizer.zero_grad()
        _, esol_preds = model(input_ids, attn_mask)
        loss_esol = mse_loss(esol_preds.squeeze(), esol_labels)
        loss_esol.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss_esol.item()
        steps += 2

    avg_loss = total_loss / steps

    # evaluate BBBP
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in bbbp_test_loader:
            input_ids = batch['input_ids'].to('cuda')
            attn_mask = batch['attention_mask'].to('cuda')
            bbbp_logits, _ = model(input_ids, attn_mask)
            probs = torch.sigmoid(bbbp_logits.squeeze())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(batch['bbbp_label'].numpy())

    bbbp_auc = roc_auc_score(all_labels, all_probs)

    # evaluate ESOL
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in esol_test_loader:
            input_ids = batch['input_ids'].to('cuda')
            attn_mask = batch['attention_mask'].to('cuda')
            _, esol_preds = model(input_ids, attn_mask)
            all_preds.extend(esol_preds.squeeze().cpu().numpy())
            all_targets.extend(batch['esol_label'].numpy())

    esol_rmse = np.sqrt(np.mean((np.array(all_preds) - np.array(all_targets))**2))

    print(f"Epoch {epoch+7}/{EXTRA_EPOCHS+6} | Loss: {avg_loss:.4f} | BBBP AUC: {bbbp_auc:.4f} | ESOL RMSE: {esol_rmse:.4f}")

    if bbbp_auc > best_bbbp_auc:
        best_bbbp_auc = bbbp_auc
        torch.save(model.state_dict(), f'{SAVE_DIR}/multitask_best.pt')
        print(f"  → Best saved (AUC: {best_bbbp_auc:.4f}) ✓")

print(f"\nFinal Best BBBP AUC: {best_bbbp_auc:.4f}")
print(f"ChemBERTa single:    0.9339")
print(f"RF Baseline:         0.9330")

Epoch 7/12 | Loss: 0.4884 | BBBP AUC: 0.9212 | ESOL RMSE: 0.9509
  → Best saved (AUC: 0.9212) ✓
Epoch 8/12 | Loss: 0.5012 | BBBP AUC: 0.9225 | ESOL RMSE: 0.9353
  → Best saved (AUC: 0.9225) ✓
Epoch 9/12 | Loss: 0.4333 | BBBP AUC: 0.9232 | ESOL RMSE: 0.9366
  → Best saved (AUC: 0.9232) ✓
Epoch 10/12 | Loss: 0.4397 | BBBP AUC: 0.9232 | ESOL RMSE: 0.9366
Epoch 11/12 | Loss: 0.4488 | BBBP AUC: 0.9232 | ESOL RMSE: 0.9366
Epoch 12/12 | Loss: 0.4594 | BBBP AUC: 0.9232 | ESOL RMSE: 0.9366

Final Best BBBP AUC: 0.9232
ChemBERTa single:    0.9339
RF Baseline:         0.9330


In [12]:
# fresh multi-task model with better architecture
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import roc_auc_score
from torch.nn import BCEWithLogitsLoss, MSELoss
import numpy as np, torch, os

class MultiTaskChemBERTaV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")
        hidden = self.backbone.config.hidden_size  # 768

        # bigger heads + batch norm
        self.bbbp_head = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
        self.esol_head = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.bbbp_head(cls), self.esol_head(cls)

model_v2 = MultiTaskChemBERTaV2().to('cuda')
print(f"Model V2 ready ✓  Params: {sum(p.numel() for p in model_v2.parameters()):,}")

EPOCHS = 12
LR     = 5e-6              # lower — stop oscillation
SAVE_DIR = '/content/drive/MyDrive/mol_predictor/checkpoints'

optimizer = AdamW(model_v2.parameters(), lr=LR, weight_decay=0.01)
total_steps = (len(bbbp_train_loader) + len(esol_train_loader)) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

bce_loss = BCEWithLogitsLoss()
mse_loss = MSELoss()
best_bbbp_auc = 0

for epoch in range(EPOCHS):
    model_v2.train()
    total_loss = 0
    steps = 0

    for bbbp_batch, esol_batch in zip(bbbp_train_loader, esol_train_loader):

        # ── combined loss — one backward pass ──
        optimizer.zero_grad()

        # BBBP
        bbbp_logits, _ = model_v2(
            bbbp_batch['input_ids'].to('cuda'),
            bbbp_batch['attention_mask'].to('cuda')
        )
        loss_bbbp = bce_loss(bbbp_logits.squeeze(), bbbp_batch['bbbp_label'].to('cuda'))

        # ESOL
        _, esol_preds = model_v2(
            esol_batch['input_ids'].to('cuda'),
            esol_batch['attention_mask'].to('cuda')
        )
        loss_esol = mse_loss(esol_preds.squeeze(), esol_batch['esol_label'].to('cuda'))

        # combined — weighted sum
        loss = loss_bbbp + 0.1 * loss_esol
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_v2.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        steps += 1

    avg_loss = total_loss / steps

    # evaluate BBBP
    model_v2.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in bbbp_test_loader:
            bbbp_logits, _ = model_v2(
                batch['input_ids'].to('cuda'),
                batch['attention_mask'].to('cuda')
            )
            probs = torch.sigmoid(bbbp_logits.squeeze())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(batch['bbbp_label'].numpy())

    bbbp_auc = roc_auc_score(all_labels, all_probs)

    # evaluate ESOL
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in esol_test_loader:
            _, esol_preds = model_v2(
                batch['input_ids'].to('cuda'),
                batch['attention_mask'].to('cuda')
            )
            all_preds.extend(esol_preds.squeeze().cpu().numpy())
            all_targets.extend(batch['esol_label'].numpy())

    esol_rmse = np.sqrt(np.mean((np.array(all_preds) - np.array(all_targets))**2))

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | BBBP AUC: {bbbp_auc:.4f} | ESOL RMSE: {esol_rmse:.4f}")

    if bbbp_auc > best_bbbp_auc:
        best_bbbp_auc = bbbp_auc
        torch.save(model_v2.state_dict(), f'{SAVE_DIR}/multitask_v2_best.pt')
        print(f"  → Best saved (AUC: {best_bbbp_auc:.4f}) ✓")

print(f"\nV2 Best BBBP AUC:    {best_bbbp_auc:.4f}")
print(f"V1 Multi-task AUC:   0.9232")
print(f"ChemBERTa single:    0.9339")
print(f"RF Baseline:         0.9330")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: seyonec/ChemBERTa-zinc-base-v1
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model V2 ready ✓  Params: 44,531,714
Epoch 1/12 | Loss: 2.0999 | BBBP AUC: 0.6506 | ESOL RMSE: 3.7137
  → Best saved (AUC: 0.6506) ✓
Epoch 2/12 | Loss: 1.9134 | BBBP AUC: 0.6711 | ESOL RMSE: 3.4585
  → Best saved (AUC: 0.6711) ✓
Epoch 3/12 | Loss: 1.6684 | BBBP AUC: 0.6838 | ESOL RMSE: 3.0966
  → Best saved (AUC: 0.6838) ✓
Epoch 4/12 | Loss: 1.3852 | BBBP AUC: 0.7182 | ESOL RMSE: 2.7044
  → Best saved (AUC: 0.7182) ✓
Epoch 5/12 | Loss: 1.1820 | BBBP AUC: 0.7848 | ESOL RMSE: 2.4870
  → Best saved (AUC: 0.7848) ✓
Epoch 6/12 | Loss: 1.0720 | BBBP AUC: 0.8071 | ESOL RMSE: 2.3876
  → Best saved (AUC: 0.8071) ✓
Epoch 7/12 | Loss: 0.9816 | BBBP AUC: 0.8223 | ESOL RMSE: 2.3321
  → Best saved (AUC: 0.8223) ✓
Epoch 8/12 | Loss: 0.9464 | BBBP AUC: 0.8289 | ESOL RMSE: 2.2868
  → Best saved (AUC: 0.8289) ✓
Epoch 9/12 | Loss: 0.9342 | BBBP AUC: 0.8431 | ESOL RMSE: 2.2499
  → Best saved (AUC: 0.8431) ✓
Epoch 10/12 | Loss: 0.8817 | BBBP AUC: 0.8582 | ESOL RMSE: 2.2206
  → Best saved (AUC: 0.8582) ✓
Ep

In [13]:
# V3 — back to V1 approach, LR 1.5e-5, 15 epochs
import torch.nn as nn
from transformers import AutoModel
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import roc_auc_score
from torch.nn import BCEWithLogitsLoss, MSELoss
import numpy as np, torch, os

class MultiTaskChemBERTaV3(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")
        hidden = self.backbone.config.hidden_size

        self.bbbp_head = nn.Sequential(
            nn.Linear(hidden, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1)
        )
        self.esol_head = nn.Sequential(
            nn.Linear(hidden, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.bbbp_head(cls), self.esol_head(cls)

model_v3 = MultiTaskChemBERTaV3().to('cuda')
print(f"V3 ready ✓")

EPOCHS   = 15
LR       = 1.5e-5          # between 1e-5 and 2e-5
SAVE_DIR = '/content/drive/MyDrive/mol_predictor/checkpoints'

optimizer = AdamW(model_v3.parameters(), lr=LR, weight_decay=0.01)
total_steps = (len(bbbp_train_loader) + len(esol_train_loader)) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

bce_loss = BCEWithLogitsLoss()
mse_loss = MSELoss()
best_bbbp_auc = 0

for epoch in range(EPOCHS):
    model_v3.train()
    total_loss = 0
    steps = 0

    for bbbp_batch, esol_batch in zip(bbbp_train_loader, esol_train_loader):

        # BBBP step
        optimizer.zero_grad()
        bbbp_logits, _ = model_v3(
            bbbp_batch['input_ids'].to('cuda'),
            bbbp_batch['attention_mask'].to('cuda')
        )
        loss_bbbp = bce_loss(bbbp_logits.squeeze(), bbbp_batch['bbbp_label'].to('cuda'))
        loss_bbbp.backward()
        torch.nn.utils.clip_grad_norm_(model_v3.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss_bbbp.item()

        # ESOL step
        optimizer.zero_grad()
        _, esol_preds = model_v3(
            esol_batch['input_ids'].to('cuda'),
            esol_batch['attention_mask'].to('cuda')
        )
        loss_esol = mse_loss(esol_preds.squeeze(), esol_batch['esol_label'].to('cuda'))
        loss_esol.backward()
        torch.nn.utils.clip_grad_norm_(model_v3.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss_esol.item()
        steps += 2

    avg_loss = total_loss / steps

    # evaluate BBBP
    model_v3.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in bbbp_test_loader:
            bbbp_logits, _ = model_v3(
                batch['input_ids'].to('cuda'),
                batch['attention_mask'].to('cuda')
            )
            probs = torch.sigmoid(bbbp_logits.squeeze())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(batch['bbbp_label'].numpy())

    bbbp_auc = roc_auc_score(all_labels, all_probs)

    # evaluate ESOL
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in esol_test_loader:
            _, esol_preds = model_v3(
                batch['input_ids'].to('cuda'),
                batch['attention_mask'].to('cuda')
            )
            all_preds.extend(esol_preds.squeeze().cpu().numpy())
            all_targets.extend(batch['esol_label'].numpy())

    esol_rmse = np.sqrt(np.mean((np.array(all_preds) - np.array(all_targets))**2))

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | BBBP AUC: {bbbp_auc:.4f} | ESOL RMSE: {esol_rmse:.4f}")

    if bbbp_auc > best_bbbp_auc:
        best_bbbp_auc = bbbp_auc
        torch.save(model_v3.state_dict(), f'{SAVE_DIR}/multitask_v3_best.pt')
        print(f"  → Best saved (AUC: {best_bbbp_auc:.4f}) ✓")

print(f"\nV3 Best BBBP AUC:  {best_bbbp_auc:.4f}")
print(f"V1 Multi-task:     0.9232")
print(f"ChemBERTa single:  0.9339")
print(f"RF Baseline:       0.9330")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: seyonec/ChemBERTa-zinc-base-v1
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


V3 ready ✓
Epoch 1/15 | Loss: 6.2774 | BBBP AUC: 0.7012 | ESOL RMSE: 3.0093
  → Best saved (AUC: 0.7012) ✓
Epoch 2/15 | Loss: 3.1370 | BBBP AUC: 0.8475 | ESOL RMSE: 1.9121
  → Best saved (AUC: 0.8475) ✓
Epoch 3/15 | Loss: 1.4881 | BBBP AUC: 0.9037 | ESOL RMSE: 1.4129
  → Best saved (AUC: 0.9037) ✓
Epoch 4/15 | Loss: 1.0036 | BBBP AUC: 0.9178 | ESOL RMSE: 1.2479
  → Best saved (AUC: 0.9178) ✓
Epoch 5/15 | Loss: 0.8234 | BBBP AUC: 0.9241 | ESOL RMSE: 1.1204
  → Best saved (AUC: 0.9241) ✓
Epoch 6/15 | Loss: 0.7054 | BBBP AUC: 0.9217 | ESOL RMSE: 1.0559
Epoch 7/15 | Loss: 0.5770 | BBBP AUC: 0.9300 | ESOL RMSE: 1.0206
  → Best saved (AUC: 0.9300) ✓
Epoch 8/15 | Loss: 0.5170 | BBBP AUC: 0.9280 | ESOL RMSE: 0.9809
Epoch 9/15 | Loss: 0.4800 | BBBP AUC: 0.9393 | ESOL RMSE: 0.8886
  → Best saved (AUC: 0.9393) ✓
Epoch 10/15 | Loss: 0.4109 | BBBP AUC: 0.9213 | ESOL RMSE: 0.8725
Epoch 11/15 | Loss: 0.3660 | BBBP AUC: 0.9272 | ESOL RMSE: 0.8622
Epoch 12/15 | Loss: 0.3629 | BBBP AUC: 0.9279 | ESOL RM